In [143]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from ucimlrepo import fetch_ucirepo 

In [60]:
data = fetch_ucirepo(id=222) 

In [61]:
X = data.data.features
y = data.data.targets
meta = data.metadata
variables = data.variables

Общее опсиание датасета: маркетинговые кампании португальского банка.

Маркетинговые кампании проводились с помощью телефонных звонков. Во многих случаях требовалось несколько контактов с одним и тем же клиентом, чтобы определить, оформит-ли клиент банковский срочный депозит.

In [62]:
X.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [63]:
y.head()

,y
0,no
1,no
2,no
3,no
4,no


In [64]:
data = X.merge(y,left_index=True, right_index=True)

In [65]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [94]:
variables

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,None,no
1,job,Feature,Categorical,Occupation,"type of job (categorical: 'admin.','blue-colla...",None,no
2,marital,Feature,Categorical,Marital Status,"marital status (categorical: 'divorced','marri...",None,no
3,education,Feature,Categorical,Education Level,"(categorical: 'basic.4y','basic.6y','basic.9y'...",None,no
4,default,Feature,Binary,None,has credit in default?,None,no
5,balance,Feature,Integer,None,average yearly balance,euros,no
6,housing,Feature,Binary,None,has housing loan?,None,no
7,loan,Feature,Binary,None,has personal loan?,None,no
8,contact,Feature,Categorical,None,contact communication type (categorical: 'cell...,None,yes
9,day_of_week,Feature,Date,None,last contact day of the week,None,no


In [93]:
variables[['name', 'description']]

,name,description
0,age,None
1,job,"type of job (categorical: 'admin.','blue-colla..."
2,marital,"marital status (categorical: 'divorced','marri..."
3,education,"(categorical: 'basic.4y','basic.6y','basic.9y'..."
4,default,has credit in default?
5,balance,average yearly balance
6,housing,has housing loan?
7,loan,has personal loan?
8,contact,contact communication type (categorical: 'cell...
9,day_of_week,last contact day of the week


Описание признаков(сорри за мой кривой перевод с английского):

- age — возраст клиента
- job — должность клиента
- marital — семейное положение клиента
- education — уровень образования клиента
- default — имеет просроченный кредит?
- balance — средний годовой баланс клиента
- housing — есть ли ипотека?
- loan — есть ли потребительский кредит?
- contact — средство связи с клиентом
- day_of_week — число месяцв последнего контакта с клиентом(почему-то в датасете назвали день недели, но дальше выяснилось, что это день месяца)
- month — месяц последнего контакта с клиентом
- duration — длительность последнего контакта в секундах
- campaign — количество контактов с клиентом в рамках текущей кампании
- pdays — количество дней после предыдущего контакта с клиентом из прошлой кампании
- previous — количество контактов с клиентом до текущей кампании
- poutcome — результат предыдущей маркетинговой кампании
- y — оформил ли клиент срочный депозит

Посмотрим пропуски

In [66]:
data.isna().sum()

age                0
job              288
marital            0
education       1857
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
y                  0
dtype: int64

Заполним пропуски

In [68]:
columns_names = list(data.columns)

In [79]:
unique_values = {}

for x in columns_names:
    unique_values[x] = data[x].unique()

In [80]:
unique_values['poutcome']

array([nan, 'failure', 'other', 'success'], dtype=object)

In [81]:
data['poutcome'] = data['poutcome'].fillna('unknown')

In [82]:
unique_values['job']

array(['management', 'technician', 'entrepreneur', 'blue-collar', nan,
       'retired', 'admin.', 'services', 'self-employed', 'unemployed',
       'housemaid', 'student'], dtype=object)

In [83]:
data['job'] = data['job'].fillna('unknown')

In [84]:
unique_values['education']

array(['tertiary', 'secondary', nan, 'primary'], dtype=object)

In [85]:
data['education'] = data['education'].fillna('unknown')

In [86]:
unique_values['contact']

array([nan, 'cellular', 'telephone'], dtype=object)

In [87]:
data['contact'] = data['contact'].fillna('unknown')

In [95]:
data.isna().sum().sum()

np.int64(0)

Все, пропуски заполнены, можно начинать рисовать графики

In [97]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [102]:
fig = px.histogram(data, x='y', title='Распределние целевой переменной')

fig.show()

In [122]:
(data['y'] == 'no').sum()/len(data['y'])

np.float64(0.8830151954170445)

88% ответили нет и 12% ответили да. У нас несбаласированные классы.

In [107]:
fig = px.box(data, x='age', title=f'Распределние возраста')
fig.show()

- медиана - 39;
- основной диапазон 33-48;
- правая асимметрия;
- выбросы после 70 лет.

In [108]:
fig = px.box(data, x='balance', title=f'Распределние денежного баланса')
fig.show()

- медиана - 448;
- основной диапазон 72-1428;
- жесткая правая асимметрия;
- выбросы после 3462 и до -1944;
- есть экстремальные значения.

In [109]:
fig = px.box(data, x='duration', title=f'Распределние продолжительности последнего звонка')
fig.show()

- медиана - 180;
- основной диапазон 103-319;
- правая асимметрия;
- выбросы после 643;

In [178]:
fig = px.box(data, x='campaign', title=f'Распределние кол-ва контактов, осуществленных в рамках текущей кампании')
fig.show()

- медиана - 2;
- основной диапазон 1-3;
- выбросы после 6;

In [179]:
fig = px.box(data, x='pdays', title=f'Распределние дней, прошедших после последнего контакта с прошлой кампании')
fig.show()

Такой график связан с тем, что было обзвонено очень много клиентов, с которыми не было контакта в прошлой кампании. В таких случаях в столбце pdays стоит значение -1. Поэтому на графике мы видим большое количество выбросов - это клиенты с которыми общались до этой кампании.

In [115]:
data['more_than_1_year_since_last_contact'] = data['pdays'] > 365

In [180]:
fig = px.box(data, x='previous', title=f'Распределние кол-ва контактов, осуществленных до текущей кампании')
fig.show()

Кого-то жестко обзванивали(275). Тут опять такая же ситуация как в прошлом графике, большинство клиентов были обзвонены в первый раз из-за этого большиснтво значений равно 0.

In [113]:
data['has_previous_contact'] = data['previous'] > 0

посмотрим корреляцию между числовыми признаками

In [123]:
num_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']

In [124]:
corr = data[num_cols].corr()

[Хитмап в плотли](https://habr.com/ru/companies/netologyru/articles/992594/)

In [127]:
px.imshow(corr, text_auto=True, title='Корреляция между числовыми признаками')

Посмотрим скрипичную диаграмму наних числовых признаков

In [128]:
for col in num_cols:
    fig = px.violin(data, x=col, title=f'Распределние {col}')
    fig.show()

In [248]:
group_by_job = data.groupby('job')['y'].value_counts().reset_index()

In [249]:
group_by_job.head()

,job,y,count
0,admin.,no,4540
1,admin.,yes,631
2,blue-collar,no,9024
3,blue-collar,yes,708
4,entrepreneur,no,1364


In [250]:
fig1 = go.Bar(
    x=group_by_job[group_by_job['y'] == 'no']['job'],
    y=group_by_job[group_by_job['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_job[group_by_job['y'] == 'yes']['job'],
    y=group_by_job[group_by_job['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по профессиям'})

fig.show()

In [151]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y,has_previous_contact,more_than_1_year_since_last_contact
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no,False,False
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no,False,False
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no,False,False
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no,False,False
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no,False,False


In [251]:
group_by_education = data.groupby('education')['y'].value_counts().reset_index()

fig1 = go.Bar(
    x=group_by_education[group_by_education['y'] == 'no']['education'],
    y=group_by_education[group_by_education['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_education[group_by_education['y'] == 'yes']['education'],
    y=group_by_education[group_by_education['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по образованию'})

fig.show()

In [ ]:
group_by_marital = data.groupby('marital')['y'].value_counts().reset_index()

fig1 = go.Bar(
    x=group_by_marital[group_by_marital['y'] == 'no']['marital'],
    y=group_by_marital[group_by_marital['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_marital[group_by_marital['y'] == 'yes']['marital'],
    y=group_by_marital[group_by_marital['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по семейному положению'})

fig.show()

In [252]:
group_by_contact = data.groupby('contact')['y'].value_counts().reset_index()

fig1 = go.Bar(
    x=group_by_contact[group_by_contact['y'] == 'no']['contact'],
    y=group_by_contact[group_by_contact['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_contact[group_by_contact['y'] == 'yes']['contact'],
    y=group_by_contact[group_by_contact['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по контактным данным'})

fig.show()

In [ ]:
group_by_month = data.groupby('month')['y'].value_counts().reset_index()

fig1 = go.Bar(
    x=group_by_month[group_by_month['y'] == 'no']['month'],
    y=group_by_month[group_by_month['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_month[group_by_month['y'] == 'yes']['month'],
    y=group_by_month[group_by_month['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по месяцам'})

fig.show()

In [253]:
group_by_poutcome = data.groupby('poutcome')['y'].value_counts().reset_index()

fig1 = go.Bar(
    x=group_by_poutcome[group_by_poutcome['y'] == 'no']['poutcome'],
    y=group_by_poutcome[group_by_poutcome['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_poutcome[group_by_poutcome['y'] == 'yes']['poutcome'],
    y=group_by_poutcome[group_by_poutcome['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по результатам предыдущих контактов'})

fig.show()

In [158]:
unique_values['day_of_week']

array([ 5,  6,  7,  8,  9, 12, 13, 14, 15, 16, 19, 20, 21, 23, 26, 27, 28,
       29, 30,  2,  3,  4, 11, 17, 18, 24, 25,  1, 10, 22, 31])

Очень странные значения для дня недели, скорее всего ошиблись названием и это день месяца, получается можно посмотреть по датам

In [177]:
fig = px.violin(data, x='day_of_week', title=f'Распределние кол-ва звонков в зависимости от дня недели')
fig.show()

In [160]:
unique_values['month']

array(['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'jan', 'feb',
       'mar', 'apr', 'sep'], dtype=object)

In [212]:
month_names_dict = {
    'jan': '01',
    'feb': '02',
    'mar': '03',
    'apr': '04',
    'may': '05',
    'jun': '06',
    'jul': '07',
    'aug': '08',
    'sep': '09',
    'oct': '10',
    'nov': '11',
    'dec': '12'
}

In [213]:
data["day_of_week"].astype(str) + data["month"].replace(month_names_dict)

0         505
1         505
2         505
3         505
4         505
         ... 
45206    1711
45207    1711
45208    1711
45209    1711
45210    1711
Length: 45211, dtype: object

In [230]:
data['date'] = data["day_of_week"].astype(str) + '.' + data["month"].replace(month_names_dict)

In [231]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,...,campaign,pdays,previous,poutcome,y,has_previous_contact,more_than_1_year_since_last_contact,date,month_num,date_str
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
1,44,technician,single,secondary,no,29,yes,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
4,33,unknown,single,unknown,no,1,no,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05


In [232]:
month_names_dict_num = {
    'jan': 1,
    'feb': 2,
    'mar': 3,
    'apr': 4,
    'may': 5,
    'jun': 6,
    'jul': 7,
    'aug': 8,
    'sep': 9,
    'oct': 10,
    'nov': 11,
    'dec': 12
}

In [233]:
data['month_num'] = data['month'].replace(month_names_dict_num)

/var/folders/h5/t5dxgffs3r172t83x2g98mv00000gn/T/ipykernel_15518/800314259.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['month_num'] = data['month'].replace(month_names_dict_num)


In [234]:
data['month_num'].head()

0    5
1    5
2    5
3    5
4    5
Name: month_num, dtype: int64

In [235]:
data['date_str'] = data["day_of_week"].astype(str) + '.' + data["month"].replace(month_names_dict)

In [254]:
group_by_date = data.groupby(['date', 'month_num', 'day_of_week'])['y'].value_counts().reset_index()

group_by_date = group_by_date.sort_values(by=['month_num', 'day_of_week'])


fig1 = go.Bar(
    x=group_by_date[group_by_date['y'] == 'no']['date'],
    y=group_by_date[group_by_date['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_date[group_by_date['y'] == 'yes']['date'],
    y=group_by_date[group_by_date['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по датам'})

fig.show()

In [255]:
group_by_date = data.groupby('date')['y'].value_counts().reset_index()


fig1 = go.Bar(
    x=group_by_date[group_by_date['y'] == 'no']['date'],
    y=group_by_date[group_by_date['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_date[group_by_date['y'] == 'yes']['date'],
    y=group_by_date[group_by_date['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по датам'})

fig.show()

In [ ]:
group_by_age = data.groupby('age')['y'].value_counts().reset_index()


fig1 = go.Bar(
    x=group_by_age[group_by_age['y'] == 'no']['age'],
    y=group_by_age[group_by_age['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_age[group_by_age['y'] == 'yes']['age'],
    y=group_by_age[group_by_age['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по возрасту'})

fig.show()

In [ ]:
group_by_default = data.groupby('default')['y'].value_counts().reset_index()

group_by_default.loc[group_by_default['default'] == 'no', 'default'] = 'нет просрочек по кредиту'
group_by_default.loc[group_by_default['default'] == 'yes', 'default'] = 'есть просрочки по кредиту'

fig1 = go.Bar(
    x=group_by_default[group_by_default['y'] == 'no']['default'],
    y=group_by_default[group_by_default['y'] == 'no']['count'],
    name='no',
)

fig2 = go.Bar(
    x=group_by_default[group_by_default['y'] == 'yes']['default'],
    y=group_by_default[group_by_default['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по наличию просрочек кредита'})

fig.show()

In [247]:
group_by_housing = data.groupby('housing')['y'].value_counts().reset_index()

group_by_housing.loc[group_by_housing['housing'] == 'no', 'housing'] = 'нет ипотеки'
group_by_housing.loc[group_by_housing['housing'] == 'yes', 'housing'] = 'есть ипотека'

fig1 = go.Bar(
    x=group_by_housing[group_by_housing['y'] == 'no']['housing'],
    y=group_by_housing[group_by_housing['y'] == 'no']['count'],
    name='no',
)

fig2 = go.Bar(
    x=group_by_housing[group_by_housing['y'] == 'yes']['housing'],
    y=group_by_housing[group_by_housing['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по наличию просрочек кредита'})

fig.show()

In [246]:
group_by_loan = data.groupby('loan')['y'].value_counts().reset_index()

group_by_loan.loc[group_by_loan['loan'] == 'no', 'loan'] = 'нет кредита'
group_by_loan.loc[group_by_loan['loan'] == 'yes', 'loan'] = 'есть кредит'

fig1 = go.Bar(
    x=group_by_loan[group_by_loan['y'] == 'no']['loan'],
    y=group_by_loan[group_by_loan['y'] == 'no']['count'],
    name='no',
)

fig2 = go.Bar(
    x=group_by_loan[group_by_loan['y'] == 'yes']['loan'],
    y=group_by_loan[group_by_loan['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по наличию просрочек кредита'})

fig.show()

In [256]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,...,campaign,pdays,previous,poutcome,y,has_previous_contact,more_than_1_year_since_last_contact,date,month_num,date_str
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
1,44,technician,single,secondary,no,29,yes,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05
4,33,unknown,single,unknown,no,1,no,no,unknown,5,...,1,-1,0,unknown,no,False,False,5.05,5,5.05


In [ ]:
group_by_has_previous_contact = data.groupby('has_previous_contact')['y'].value_counts().reset_index()

group_by_has_previous_contact.loc[group_by_has_previous_contact['has_previous_contact'] == False, 'has_previous_contact'] = 'не было контактов'
group_by_has_previous_contact.loc[group_by_has_previous_contact['has_previous_contact'] == True, 'has_previous_contact'] = 'были контакты'

fig1 = go.Bar(
    x=group_by_has_previous_contact[group_by_has_previous_contact['y'] == 'no']['has_previous_contact'],
    y=group_by_has_previous_contact[group_by_has_previous_contact['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_has_previous_contact[group_by_has_previous_contact['y'] == 'yes']['has_previous_contact'],
    y=group_by_has_previous_contact[group_by_has_previous_contact['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по наличию предыдущих контактов'})

fig.show()

/var/folders/h5/t5dxgffs3r172t83x2g98mv00000gn/T/ipykernel_15518/4121843293.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'не было контактов' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  group_by_has_previous_contact.loc[group_by_has_previous_contact['has_previous_contact'] == False, 'has_previous_contact'] = 'не было контактов'


In [279]:
data['interest_in_campaign'] = data['campaign'].apply(
    lambda x: 
    '1 звонок' if x == 1 else
    '2 звонка' if x == 2 else
    '3 звонка' if x == 3 else
    '4 звонка' if x == 4 else
    '5 и более звонков' if x > 4 else
    'нет звонков'
    )

In [280]:
data['interest_in_campaign'].unique()

array(['1 звонок', '2 звонка', '3 звонка', '5 и более звонков',
       '4 звонка'], dtype=object)

In [281]:
data['campaign'].unique()

array([ 1,  2,  3,  5,  4,  6,  7,  8,  9, 10, 11, 12, 13, 19, 14, 24, 16,
       32, 18, 22, 15, 17, 25, 21, 43, 51, 63, 41, 26, 28, 55, 50, 38, 23,
       20, 29, 31, 37, 30, 46, 27, 58, 33, 35, 34, 36, 39, 44])

In [282]:
unique_values['campaign']

array([ 1,  2,  3,  5,  4,  6,  7,  8,  9, 10, 11, 12, 13, 19, 14, 24, 16,
       32, 18, 22, 15, 17, 25, 21, 43, 51, 63, 41, 26, 28, 55, 50, 38, 23,
       20, 29, 31, 37, 30, 46, 27, 58, 33, 35, 34, 36, 39, 44])

In [ ]:
group_by_interest_in_campaign = data.groupby('interest_in_campaign')['y'].value_counts().reset_index()

fig1 = go.Bar(
    x=group_by_interest_in_campaign[group_by_interest_in_campaign['y'] == 'no']['interest_in_campaign'],
    y=group_by_interest_in_campaign[group_by_interest_in_campaign['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_interest_in_campaign[group_by_interest_in_campaign['y'] == 'yes']['interest_in_campaign'],
    y=group_by_interest_in_campaign[group_by_interest_in_campaign['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по наличию предыдущих контактов'})

fig.show()

In [293]:
unique_values['duration'].sort()

In [294]:
unique_values['duration']

array([   0,    1,    2, ..., 3785, 3881, 4918], shape=(1573,))

In [299]:
(data['duration'] == 0).sum()

np.int64(3)

In [ ]:
data['duration_group'] = data['duration'].apply(
    lambda x: 
    'менее 1 минуты' if x > 0 and x < 60 else
    '1 - 3 минуты' if x >= 60 and x < 180 else
    'более 3 минут' if x >= 180 else
    'звонок сбросили'
    )

In [305]:
group_by_duration_group = data.groupby(['duration_group'])['y'].value_counts().reset_index()

group_by_duration_group = group_by_duration_group[group_by_duration_group['duration_group'] != 'звонок сбросили'].sort_values(by = 'count')

fig1 = go.Bar(
    x=group_by_duration_group[group_by_duration_group['y'] == 'no']['duration_group'],
    y=group_by_duration_group[group_by_duration_group['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_duration_group[group_by_duration_group['y'] == 'yes']['duration_group'],
    y=group_by_duration_group[group_by_duration_group['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по времени последнего звонка'})

fig.show()

In [306]:
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,...,y,has_previous_contact,more_than_1_year_since_last_contact,date,month_num,date_str,was_successfully_contacted_before,interest_in_campaign,answered_calls,duration_group
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,...,no,False,False,5.05,5,5.05,False,1 звонок,True,более 3 минут
1,44,technician,single,secondary,no,29,yes,no,unknown,5,...,no,False,False,5.05,5,5.05,False,1 звонок,True,1 - 3 минуты
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,...,no,False,False,5.05,5,5.05,False,1 звонок,True,1 - 3 минуты
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,...,no,False,False,5.05,5,5.05,False,1 звонок,True,1 - 3 минуты
4,33,unknown,single,unknown,no,1,no,no,unknown,5,...,no,False,False,5.05,5,5.05,False,1 звонок,True,более 3 минут


In [309]:
data['has_any_type_of_loan'] = (data['housing'] == 'yes') | (data['loan'] == 'yes') | (data['default'] == 'yes')

In [315]:
group_by_any_loan = data.groupby(['has_any_type_of_loan'])['y'].value_counts().reset_index()

group_by_any_loan.loc[group_by_any_loan['has_any_type_of_loan'] == False, 'has_any_type_of_loan'] = 'нет кредитов'
group_by_any_loan.loc[group_by_any_loan['has_any_type_of_loan'] == True, 'has_any_type_of_loan'] = 'есть кредиты'

fig1 = go.Bar(
    x=group_by_any_loan[group_by_any_loan['y'] == 'no']['has_any_type_of_loan'],
    y=group_by_any_loan[group_by_any_loan['y'] == 'no']['count'],
    name='no'
)

fig2 = go.Bar(
    x=group_by_any_loan[group_by_any_loan['y'] == 'yes']['has_any_type_of_loan'],
    y=group_by_any_loan[group_by_any_loan['y'] == 'yes']['count'],
    name='yes'
)

fig = go.Figure(data=[fig1, fig2], layout={'title': 'Распределение целевой переменной по кредитам'})

fig.show()

/var/folders/h5/t5dxgffs3r172t83x2g98mv00000gn/T/ipykernel_15518/3261409062.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'нет кредитов' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  group_by_any_loan.loc[group_by_any_loan['has_any_type_of_loan'] == False, 'has_any_type_of_loan'] = 'нет кредитов'
